In [1]:
import os
import bidict
import pickle
import duckdb

import pandas as pd
import polars as pl

from orqa.utils import sanitize_string

In [2]:
tables_path     = '../data/datasets/CAN/tables/tables_from10000_to15000'
metadata_path   = '../data/datasets/CAN/metadata/metadata_from10000_to15000'

db_path         = '../data/datasets/CAN/database/CAN.db'
valdict_path    = '../data/datasets/CAN/database/values_dict.pickle'

In [33]:
table_ids = list(sorted(os.listdir(tables_path), reverse=True))
len(table_ids)

3689

In [4]:
with open(valdict_path, 'rb') as fr:
    values = pickle.load(fr)

In [5]:
len(values.values()), len(set(values.values()))

(10350000, 10350000)

In [6]:
values_bidict = bidict.bidict(values)

In [7]:
len(values_bidict)

10350000

In [9]:
from itertools import chain

def get_table_values(table_id):
    df = pl.read_parquet(f'{tables_path}/{table_id}')
    df = df[[s.name for s in df if not (s.null_count() == df.height)]]

    return set(
        filter(
            lambda v: v not in values, 
            map(
                lambda s: sanitize_string(str(s)), 
                chain(*(
                    df.select(col).drop_nans().drop_nulls().unique().get_column(col).to_list() 
                    for col in df.columns
                    )
                )                
            )
        )
    )

In [10]:
con = duckdb.connect(db_path, read_only=True)

In [94]:
i_tab = 6
df = pl.read_parquet(f"{tables_path}/{table_ids[i_tab]}")
df

"ï»¿""REF_DATE""",GEO,DGUID,Age,Type of institution attended,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
str,str,str,str,str,str,i64,str,i64,str,str,f64,str,f64,f64,i64
"""1995/1996""","""Canada""","""2021A000011124""","""15 years old""","""Total type of institution atte…","""Percent""",239,"""units""",0,"""v99908352""","""1.1.1""",97.0,"""A""",null,null,0
"""1995/1996""","""Canada""","""2021A000011124""","""15 years old""","""Elementary/High School""","""Percent""",239,"""units""",0,"""v99908353""","""1.1.2""",97.0,"""A""",null,null,0
"""1995/1996""","""Canada""","""2021A000011124""","""15 years old""","""College""","""Percent""",239,"""units""",0,"""v99908354""","""1.1.3""",null,"""x""",null,null,0
"""1995/1996""","""Canada""","""2021A000011124""","""15 years old""","""University""","""Percent""",239,"""units""",0,"""v99908355""","""1.1.4""",null,"""x""",null,null,0
"""1995/1996""","""Canada""","""2021A000011124""","""16 years old""","""Total type of institution atte…","""Percent""",239,"""units""",0,"""v99908356""","""1.2.1""",94.0,"""A""",null,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2023/2024""","""Canada""","""2021A000011124""","""28 years old""","""University""","""Percent""",239,"""units""",0,"""v99908407""","""1.14.4""",6.0,"""B""",null,null,0
"""2023/2024""","""Canada""","""2021A000011124""","""29 years old""","""Total type of institution atte…","""Percent""",239,"""units""",0,"""v99908408""","""1.15.1""",7.0,"""B""",null,null,0
"""2023/2024""","""Canada""","""2021A000011124""","""29 years old""","""Elementary/High School""","""Percent""",239,"""units""",0,"""v99908409""","""1.15.2""",null,"""x""",null,null,0


In [98]:
for i_col in range(len(df.columns)):
    print(i_col, df.columns[i_col])
    qcol = df.get_columns()[i_col].unique().drop_nulls().to_list()
    q = list(map(lambda s: str(values_bidict[s]), map(lambda s: sanitize_string(str(s)), qcol)))
    if not q: continue
    res = con.sql(f"""
            SELECT TableId, ColumnId, COUNT(DISTINCT CellValue) AS intersec FROM AllTables
            WHERE CellValue IN ({','.join(q)})
            AND TableId <> {i_tab}
            GROUP BY TableId, ColumnId
            ORDER BY COUNT(DISTINCT CellValue) DESC
            LIMIT 5;
    """).fetchall()
    print(res)

0 ï»¿"REF_DATE"
[(12, 0, 26), (9, 0, 25)]
1 GEO
[(19, 0, 1), (27, 0, 1), (44, 5, 1), (2, 5, 1), (20, 0, 1)]
2 DGUID
[(12, 2, 1), (72, 2, 1)]
3 Age
[]
4 Type of institution attended
[]
5 UOM
[(12, 6, 1), (79, 7, 1), (49, 6, 1), (72, 8, 1)]
6 UOM_ID
[(57, 6, 1), (12, 7, 1), (57, 1, 1), (59, 4, 1), (42, 22, 1)]
7 SCALAR_FACTOR
[(72, 10, 1), (12, 8, 1), (49, 8, 1), (96, 6, 1)]
8 SCALAR_ID
[(18, 6, 1), (14, 10, 1), (31, 2, 1), (85, 4, 1), (32, 3, 1)]
9 VECTOR
[]
10 COORDINATE
[]
11 VALUE
[(76, 12, 89), (12, 12, 88), (72, 14, 74), (49, 12, 45), (94, 3, 35)]
12 STATUS
[(1, 9, 6), (1, 7, 6), (1, 15, 6), (1, 17, 6), (1, 5, 6)]
13 SYMBOL
14 TERMINATED
15 DECIMALS
[(34, 4, 1), (46, 6, 1), (17, 6, 1), (31, 10, 1), (56, 3, 1)]


In [60]:
rdf = pl.read_parquet(f"{tables_path}/{table_ids[46]}")
rdf

Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
str,str,str,str,str,str,str,str,str,str,f64,f64
"""Table 43a""","""Employees who suffered a disab…",null,null,null,null,null,null,null,null,null,null
"""Size of the company""","""Age group""","""Proportion of industrial compa…","""Quality indicator""","""Number of employees""",null,null,null,null,null,null,null
null,null,null,null,"""Male""","""Quality indicator""","""Female""","""Quality indicator""","""Total""","""Quality indicator""",null,null
null,null,"""%""","""SE""","""SUM""","""CV""","""SUM""","""CV""","""SUM""","""CV""",null,null
"""All sizes""","""Total""","""10.7""","""A""","""14,218""","""B""","""6,798""","""C""","""21,018""","""B""",null,null
…,…,…,…,…,…,…,…,…,…,…,…
"""Symbols: """,null,null,null,null,null,null,null,null,null,null,null
""" not applicable""",null,null,null,null,null,null,null,null,null,null,null
"""x suppressed to meet the conf…",null,null,null,null,null,null,null,null,null,null,null


In [70]:
df.select(pl.nth(i_col)).unique().drop_nulls().join(
    rdf.select(pl.nth(0)).unique().drop_nulls(),
    left_on='Unnamed: 0', right_on='Unnamed: 0'
)

Unnamed: 0
str
"""Symbols: """
"""Coefficient of variation (CV)"""
"""Air transportation"""
"""Other"""
"""Maritime transportation"""
…
"""Components may not add up to t…"
"""Rail transportation"""
"""Totals and sub-totals have bee…"


In [68]:
rdf.select(pl.nth(0)).unique().drop_nulls()

Unnamed: 0
str
"""Other"""
"""Source: """
""" not applicable"""
"""F = Greater than 25.00% too …"
"""All sizes"""
…
"""Table 43b"""
"""100 employees and more"""
"""Air transportation"""


In [ ]:
rcol = rdf.get_columns()[1].drop_nulls().to_list()
set(rcol).intersection(qcol)

set()

In [ ]:
# con.sql("SELECT DISTINCT(TableID) FROM OrqaIndex ORDER BY TableID LIMIT 5")
con.sql("SELECT * FROM AllTables WHERE TableId = 0 LIMIT 100")

┌─────────┬──────────┬───────┬───────────┐
│ TableId │ ColumnId │ RowId │ CellValue │
│  int32  │  int32   │ int32 │   int32   │
├─────────┼──────────┼───────┼───────────┤
│       0 │        0 │     0 │        48 │
│       0 │        0 │     1 │        48 │
│       0 │        0 │     2 │        48 │
│       0 │        0 │     3 │        48 │
│       0 │        0 │     4 │        48 │
│       0 │        0 │     5 │        48 │
│       0 │        0 │     6 │        48 │
│       0 │        0 │     7 │        48 │
│       0 │        0 │     8 │        48 │
│       0 │        0 │     9 │        48 │
│       · │        · │     · │         · │
│       · │        · │     · │         · │
│       · │        · │     · │         · │
│       0 │        0 │    30 │        48 │
│       0 │        0 │    31 │        48 │
│       0 │        0 │    32 │        48 │
│       0 │        0 │    33 │        48 │
│       0 │        0 │    34 │        48 │
│       0 │        0 │    35 │        48 │
│       0 │

In [32]:
values_bidict.inverse[19]

'total_count___adults'

In [41]:
pl.read_parquet(f"{tables_path}/{table_ids[0]}")

Category,Average Daily Count - Nova Scotia,Average Daily Count,Year
str,str,i64,str
"""Adult""","""Remand - adult""",227,"""2014-2015"""
"""Adult""","""Sentenced provincial custody -…",251,"""2014-2015"""
"""Adult""","""Other - adult""",22,"""2014-2015"""
"""Adult""","""Total count - adults""",500,"""2014-2015"""
"""Adult""","""Remand - adult""",223,"""2015-2016"""
…,…,…,…
"""Youth""","""Total count - youth""",7,"""2022-2023"""
"""Youth""","""Remand - youth""",6,"""2023-2024"""
"""Youth""","""Sentenced provincial custody -…",6,"""2023-2024"""
